In [1]:
### automatically refresh the buffer
%load_ext autoreload
%autoreload 2

### solve the auto-complete issue

%config Completer.use_jedi = False
%matplotlib inline

import warnings
warnings.filterwarnings('ignore')
warnings.simplefilter(action='ignore', category=FutureWarning)

### lvl 2 setups (systerm)
import os
import numpy as np
import pandas as pd
import xarray as xr

import matplotlib as mpl
import cartopy.crs as ccrs
import cartopy.feature as cfeature

import cartopy.feature as cfeature
from cartopy.mpl.ticker import LongitudeFormatter, LatitudeFormatter
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap,LinearSegmentedColormap,BoundaryNorm
import matplotlib.dates as mdates
import geopandas as gpd
from shapely.geometry import Point
from datetime import datetime
import numpy as np

np.set_printoptions(suppress=True)

## cloud height resample from the specific time point (no interplation) for ena
- if you want SGP, just replace any 'ENA' to 'SGP'

In [ ]:
import xarray as xr
import pandas as pd
import os

# input and output directories
in_dir = "/data/shared_data/ARM_data/ENA/others/enaarsclkazrbnd1kolliasC1"
out_dir = "/data/ggong/ARM_monthly/ENA/arsclkazrbnd1kolliasC1"
os.makedirs(out_dir, exist_ok=True)

# variables to save
vars_to_save = ["cloud_layer_base_height", "cloud_layer_top_height"]

# month range
months = pd.date_range("2024-01-01", "2025-12-31", freq="MS")

for month in months:
    year = month.strftime("%Y")
    ym = month.strftime("%Y%m")

    # construct input file path pattern
    pattern = os.path.join(in_dir, f"enaarsclkazrbnd1kolliasC1.c0.{ym}*.nc")
    
    try:
        # open all files for the current month
        ds = xr.open_mfdataset(pattern, combine="by_coords")

        # keep only the required variables
        ds_sel = ds[vars_to_save]

        # output filename
        out_file = os.path.join(out_dir, f"enaarsclkazrbnd1kolliasC1_{ym}.nc")

        # save as NetCDF
        ds_sel.to_netcdf(out_file)
        print(f"Saved successfully: {out_file}")

    except FileNotFoundError:
        print(f"Missing files: {pattern}")
    except ValueError:
        # if no files are matched for the current month
        print(f"No files found for this month: {pattern}")

In [3]:
ds_c_h = xr.open_mfdataset('/data/ggong/ARM_monthly/ENA/arsclkazrbnd1kolliasC1/*.nc')

In [5]:
ds = ds_c_h.sortby("time").copy()

# Mask invalid placeholder values before further processing
for v in ["cloud_layer_base_height", "cloud_layer_top_height"]:
    ds[v] = ds[v].where((ds[v] >= 0) & (ds[v] <= 25000))

# Create a strict 2-min time grid aligned to 2-min boundaries
t0 = pd.to_datetime(ds.time.values[0])
t1 = pd.to_datetime(ds.time.values[-1])

start = t0.floor("2min")   # e.g., 00:00:00
end   = t1.floor("2min")

new_time = pd.date_range(start, end, freq="2min")

# Reindex only aligns timestamps; it does not interpolate.
# Missing 2-min timestamps are filled with NaN.
ds_2min = ds.reindex(time=new_time)

In [6]:
# ds_2min.to_netcdf('/data/ggong/ARM_monthly/ENA/cloud_height_2min_time_spot.nc')